## Libraries

In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import math 
from random import random 
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path 
import librosa 
import json 

## Set up GPU 

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device: {device}")

Device: mps


## Data

### KAGGLE GTZAN Dataset

In [3]:
from dotenv import load_dotenv 
load_dotenv() 
import kagglehub 

download_path = Path(kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification"))

print(f"download path: {download_path}")
print(f"data contents: {list(download_path.iterdir())}")

download path: /Users/pranavrajan/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1
data contents: [PosixPath('/Users/pranavrajan/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1/Data')]


## Constants 

In [7]:
SAMPLE_RATE = 22050
DURATION = 30  # measured in seconds
SAMPLES_PER_TRACK = SAMPLE_RATE * DURATION

In [4]:
def save_mfcc(dataset_path, json_path, n_mfcc=13, n_fft=2048, hop_length=512, num_segments=5):
    # dictionary to store data
    data = {
        # genres
        "mapping": [],
        # mfcc vectors for each of the segments - the training inputs
        "mfcc": [],
        # the target variables
        "labels": [],
    }
 
    num_samples_per_segment = int(SAMPLES_PER_TRACK / num_segments)
    expected_num_mfcc_vectors_per_segment = math.ceil(num_samples_per_segment / hop_length)  # 1.2 => 2
 
    # loop through all the genre folders (one level down from dataset_path)
    # sorted() makes the label order deterministic across machines
    genre_dirs = sorted(p for p in dataset_path.iterdir() if p.is_dir())
 
    for i, genre_dir in enumerate(genre_dirs):
        # save the semantic label (e.g. "blues")
        semantic_label = genre_dir.name
        data["mapping"].append(semantic_label)
        print(f"\nProcessing {semantic_label}")
 
        # process all wav files for current genre
        # .glob("*.wav") automatically skips hidden files like .DS_Store
        for file_path in sorted(genre_dir.glob("*.wav")):
            # some GTZAN files are corrupted (e.g. jazz.00054) - skip them
            try:
                signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)
            except Exception as e:
                print(f"Skipping corrupted file {file_path.name}: {e}")
                continue
 
            # process segments extracting mfcc and storing data
            # s - current segment we are in. slide to the right the number of samples per segment
            for s in range(num_segments):
                start_sample = num_samples_per_segment * s  # s=0 => 0
                end_sample = start_sample + num_samples_per_segment # s=0 => num_samples_per_segment

                # returns a matrix (mfcc, time)
                mfcc = librosa.feature.mfcc(
                    y=signal[start_sample:end_sample],
                    sr=sr,
                    n_fft=n_fft,
                    n_mfcc=n_mfcc,
                    hop_length=hop_length,
                )

                # convert to (time, mfcc) for deep learning 
                mfcc = mfcc.T
 
                # store mfcc for segment if it has the expected length
                if len(mfcc) == expected_num_mfcc_vectors_per_segment:
                    data["mfcc"].append(mfcc.tolist())
                    data["labels"].append(i)
                    print(f"{file_path}, segment: {s + 1}")
 
    # make sure the output directory exists before writing
    json_path.parent.mkdir(parents=True, exist_ok=True)
 
    with open(json_path, "w") as fp:
        json.dump(data, fp, indent=4)

In [ ]:
json_path = Path("data/mfcc_data.json")
dataset_path = download_path / "Data" / "genres_original"
save_mfcc(dataset_path, json_path, num_segments=10)

### Valerio Tutorial Setup

Valerio's solution(s) were designed when GTZAN was publicly available on the marsyas website and developed to fit his workflow. The script from the tutorial does not work because one of the files is corrupted as verified with gemini and claude. There are also many problems with the GTZAN dataset as described in the Sturm GTZAN report and Bob Sturm's extensive research into GTZAN.

My solution follows the design of Valerio's tutorial video solution(the commented out code below) using the GTZAN dataset from kaggle and also takes into account some of the findings from the Sturm report using pathlib and other modern file handling syntax in python for readability and learning. For further examples of parallelization and large scale data processing Valerio's github repo for the deep learning for audio with python lecture 12 is the best resource. 

In [5]:
# import os 
# """
# number of segments - need lots of data for training deep learning models 
# chop up each track into smaller segments and save inputs as segments (create a lot more input data)
# """
# def save_mfcc(dataset_path, json_path, n_mfcc=13, n_fft=2048, hop_length=512, num_segments=5):
#     # dictionary to store data 
#     data = {
#         # genres
#         "mapping": [],
#         # mfcc vectors for each of the segments - the training inputs 
#         "mfcc": [], 
#         # the target variables 
#         "labels": []
#     }

#     num_samples_per_segment = int(SAMPLES_PER_TRACK / num_segments)
#     expected_num_mfcc_vectors_per_segment = math.ceil(num_samples_per_segment / hop_length) # 1.2 => 2 
    

#     # loop through all the genres - analyze all the songs in a genre folder 
#     for i, (dirpath, dirnames, filenames) in enumerate(os.walk(dataset_path)):

#         # ensure that not at the root level 
#         if dirpath is not dataset_path:
#             # save the semantic label 
#             dirpath_components = dirpath.split("/") # genre/blues => ["genre", "blues"]
#             semantic_label = dirpath_components[-1]
#             data["mapping"].append(semantic_label)
#             print(f"\nProcessing {semantic_label}")

#             # process files for a specific genre 
#             for f in filenames:
#                 file_path = os.path.join(dirpath, f)
#                 signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)

#                 # process segments extracting mfcc and storing data 
#                 for s in range(num_segments):
#                     start_sample = num_samples_per_segment * s # s=0 => 0
#                     end_sample = start_sample + num_samples_per_segment # s=0 => num_samples_per_segment
                    
#                     mfcc = librosa.feature.mfcc(y=signal[start_sample:end_sample], 
#                                                sr=sr,
#                                                n_fft=n_fft,
#                                                n_mfcc=n_mfcc,
#                                                hop_length=hop_length)
#                     mfcc = mfcc.T 

#                     # store mfcc for segment if it has the expected length 
#                     if len(mfcc) == expected_num_mfcc_vectors_per_segment:
#                         data["mfcc"].append(mfcc.tolist())
#                         data["labels"].append(i - 1)
#                         print(f"{file_path}, segment: {s+1}")
#         #     pass 
        
#         # pass

#     with open(json_path, "w") as fp:
#         json.dump(data, fp, indent=4)
    

In [ ]:
# json_path = Path("data/mfcc_data.json")
# dataset_path = download_path / "Data" / "genres_original"
# save_mfcc(dataset_path, json_path, num_segments=10)